# Exploratory Data Analysis

Generates KDE plots for every numeric column, a full Pearson correlation heatmap, and computes correlation statistics between poverty rate and traffic fatality rate.

**Input:** `data/processed/merged_data.csv`  
**Output:** figures saved to `figs/eda/`

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats
import os


C:\Users\Josh\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\Josh\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
processed_data_path = "../data/processed/"
eda_figure_folder = "../figs/eda/"
os.makedirs(eda_figure_folder, exist_ok=True)

df = pd.read_csv(f"{processed_data_path}merged_data.csv")
df['poverty_rate'] = df['poverty_rate'].replace('.', float('nan')).astype(float)
df['median_hhi'] = df['median_hhi'].replace('.', float('nan')).astype(float)
df.head()


,state_fips,county_fips,poverty_rate,median_hhi,name,state_abbr,fips,ACCESS2,ARTHRITIS,BINGE,...,Hazardous Days,Max AQI,90th Percentile AQI,Median AQI,Days CO,Days NO2,Days Ozone,Days PM2.5,Days PM10,census_region
0,1,1,11.7,68857.0,Autauga County,AL,1001,10.4,28.2,15.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,South
1,1,3,10.0,74248.0,Baldwin County,AL,1003,9.5,26.4,17.8,...,0.0,90.0,60.0,43.0,0.0,0.0,108.0,239.0,0.0,South
2,1,5,25.5,45298.0,Barbour County,AL,1005,17.2,30.6,13.4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,South
3,1,7,19.4,56025.0,Bibb County,AL,1007,14.3,29.5,15.4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,South
4,1,9,12.8,64962.0,Blount County,AL,1009,13.1,28.4,16.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,South


In [3]:
for key in df.select_dtypes(include='number').columns:
    plt.figure()
    sns.kdeplot(df, x=key)
    plt.savefig(f"{eda_figure_folder}kde_{key}.png")
    plt.close()
print("KDE plots saved.")


C:\Users\Josh\AppData\Local\Temp\ipykernel_59460\2188804281.py:3: UserWarning: Dataset has 0 variance; skipping density estimate. Pass `warn_singular=False` to disable this warning.
  sns.kdeplot(df, x=key)


KDE plots saved.


In [4]:
plt.figure(figsize=(40, 20))
sns.heatmap(df.corr(method='pearson', numeric_only=True), annot=True, cmap='jet', fmt='.2f')
plt.savefig(eda_figure_folder + "heatmap.png")
plt.close()
print("Heatmap saved.")


Heatmap saved.


In [5]:
print(df.groupby('census_region')['Median AQI'].describe())

def get_pearsonr_no_nan(df, x, y):
    mask = df[x].notna() & df[y].notna()
    r, p = scipy.stats.pearsonr(df.loc[mask, x].astype(float), df.loc[mask, y])
    return r, p

def get_spearmanr_no_nan(df, x, y):
    mask = df[x].notna() & df[y].notna()
    r, p = scipy.stats.spearmanr(df.loc[mask, x].astype(float), df.loc[mask, y])
    return r, p

df['traffic_fatality_rate'] = df['total_fatalities'] / df['population'] * 100000
r, p = get_pearsonr_no_nan(df, 'poverty_rate', 'traffic_fatality_rate')
print(f"Pearson poverty vs traffic fatality: r={r:.4f}, p={p:.4f}")
r, p = get_spearmanr_no_nan(df, 'poverty_rate', 'traffic_fatality_rate')
print(f"Spearman poverty vs traffic fatality: r={r:.4f}, p={p:.4f}")
print("EDA complete.")


               count       mean        std   min    25%   50%   75%   max
census_region                                                            
Midwest        246.0  43.247967   8.035603  11.0  40.25  44.0  48.0  61.0
Northeast      118.0  40.838983   6.415753  14.0  37.00  41.0  44.0  58.0
South          351.0  42.957265   7.702756   7.0  39.00  43.0  47.5  64.0
West           259.0  37.899614  14.239832   3.0  28.00  42.0  47.0  79.0
Pearson poverty vs traffic fatality: r=0.0169, p=0.3449
Spearman poverty vs traffic fatality: r=0.2901, p=0.0000
EDA complete.
